# Biophysics Pipeline CNN

Trains `TDSConvCTC` using the **full biophysics EMG preprocessing pipeline** from
`Playground_Kai/data_preprocess.py`.

**Pipeline:**
```
ToTensor → TemporalAlignmentJitter(120) → RandomBandRotation
→ ChannelSelector (8 even-indexed channels/wrist)
→ TemporalFilter  (60 Hz notch + 4th-order Butterworth 20–450 Hz)
→ Decimator       (2× → 1000 Hz)
→ MelSpectrogram  (n_fft=256, win=64, hop=8, 32-bin Mel 20–450 Hz, log10)
→ SpecAugment     (train only)
```

**Model:** `SpectrogramNorm → MultiBandRotationInvariantMLP → TDSConvEncoder → CTC`  
**in_features:** 8 channels × 32 Mel bins = **256**  
**Checkpoint:** `Playground_Ben/checkpoints/best_biophysics_cnn.pt`

```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND  = REPO_ROOT / 'Playground_Ben'
SCRIPT      = PLAYGROUND / 'scripts/train_biophysics_cnn.py'
CHECKPOINT  = PLAYGROUND / 'checkpoints/best_biophysics_cnn.pt'
HYPERPARAMS = PLAYGROUND / 'checkpoints/best_hyperparams_raw_cnn.yaml'

print(f'Repo root: {REPO_ROOT}')
print(f'Hyperparams available: {HYPERPARAMS.exists()}')

## 2. (Optional) Hyperparameter Tuning

Two-phase search over `mlp_features`, `block_channels`, `num_blocks`, `kernel_width`, `lr`, `weight_decay`.  
Takes ~2–3 hours. Skip if `best_hyperparams_raw_cnn.yaml` already exists.

**Best found:** `lr=1.27e-3`, `weight_decay=1.92e-3`, `mlp_features=512`, `block_channels=32`, `num_blocks=3`, `kernel_width=24` → Val CER 82.41% (trial)

In [ ]:
if HYPERPARAMS.exists():
    print(f'Hyperparams already exist: {HYPERPARAMS}  — skipping tuning.')
    import yaml
    with open(HYPERPARAMS) as f:
        hp = yaml.safe_load(f)
    for k, v in hp.items():
        print(f'  {k}: {v}')
else:
    result = subprocess.run(
        [sys.executable, str(PLAYGROUND / 'scripts/hyperparam_tuner_raw_cnn.py'),
         '--coarse-trials', '20', '--coarse-epochs', '8',
         '--fine-trials', '10',   '--fine-epochs', '15'],
        cwd=str(REPO_ROOT)
    )
    print(f'Tuner exit code: {result.returncode}')

## 3. Train

Uses tuned hyperparameters if available, otherwise defaults  
(`lr=1e-3`, `mlp_features=384`, `block_channels=24`, `num_blocks=4`, `kernel_width=32`).

In [ ]:
import yaml

cmd = [sys.executable, str(SCRIPT), '--epochs', '150', '--batch-size', '32',
       '--window-length', '8000']

if HYPERPARAMS.exists():
    with open(HYPERPARAMS) as f:
        hp = yaml.safe_load(f)
    cmd += [
        '--lr',             str(hp.get('lr', 1e-3)),
        '--weight-decay',   str(hp.get('weight_decay', 1e-5)),
        '--mlp-features',   str(hp.get('mlp_features', 384)),
        '--block-channels', str(hp.get('block_channels', 24)),
        '--num-blocks',     str(hp.get('num_blocks', 4)),
        '--kernel-width',   str(hp.get('kernel_width', 32)),
    ]
    print('Using tuned hyperparameters.')
else:
    cmd += ['--lr', '1e-3']
    print('Using default hyperparameters.')

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_ROOT))
print(f'Exit code: {result.returncode}')

## 4. Inspect Checkpoint

In [ ]:
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location='cpu')
    print('Biophysics Pipeline CNN — Best Checkpoint')
    print(f'  Epoch   : {ckpt["epoch"]}')
    print(f'  Val CER : {ckpt["val_cer"]:.2f}%')
    print(f'  File    : {CHECKPOINT}')
else:
    print('Checkpoint not found — run the training cell first.')

## 5. Architecture Summary

```
Input: raw EMG (T=8000 raw samples, N, 2 bands, 16 ch) at 2000 Hz
  └─ ChannelSelector   → 8 even-indexed channels per wrist
  └─ TemporalFilter    → bandpass 20–450 Hz + 60 Hz notch
  └─ Decimator (2×)    → 1000 Hz
  └─ MelSpectrogram    → (T', N, 2, 8, 32)
  └─ SpectrogramNorm
  └─ MultiBandRotationInvariantMLP(in_features=256, mlp_features)
  └─ Flatten           → (T', N, 2×mlp_features)
  └─ TDSConvEncoder
  └─ Linear + LogSoftmax → (T'', N, 71)
```

**Mirrors the `TDSConvCTCModule` architecture exactly** but drives input through Kai's
biophysics preprocessing instead of the standard log-spectrogram pipeline.